# AI-Lake + Apache Airflow

Demonstrates scheduling AI-Lake ingest and search pipelines with Apache Airflow 2.9.

> **Prerequisite**: start the demo with `--profile airflow`:
>
> ```bash
> docker compose -f tests/docker/compose-demo.yml --profile airflow up -d
> ```
>
> Airflow UI: **http://localhost:8090** (user: `admin` / pass: `admin`)

**What this notebook covers:**

| Section | What you will learn |
|---|---|
| 1 | Connect to Airflow REST API, health check |
| 2 | List DAGs — `ailake_ingest_search` + `ailake_compaction` |
| 3 | Trigger `ailake_ingest_search` DAG run |
| 4 | Poll run status until completion |
| 5 | Inspect task logs |
| 6 | Pull XCom results (vector search + FTS) |
| 7 | Read Airflow-written data in Jupyter via `ailake.search()` |
| 8 | Trigger `ailake_compaction` DAG run |
| 9 | Direct PythonOperator demo — no Airflow running needed |
| 10 | `AilakeWriteOperator` pattern (production CLI-based approach) |

## 0. Setup

In [ ]:
import os
import json
import time
import pathlib
from urllib import request as urlrequest
from base64 import b64encode
from datetime import datetime, timezone

import numpy as np
import ailake

# Airflow REST API base URL — auto-set by compose when --profile airflow is active.
# Override with AIRFLOW_URL env var if running outside compose.
AIRFLOW_URL  = os.environ.get("AIRFLOW_URL",  "http://ailake-demo-airflow:8080")
AIRFLOW_USER = os.environ.get("AIRFLOW_USER", "admin")
AIRFLOW_PASS = os.environ.get("AIRFLOW_PASSWORD", "admin")

# Tables written by Airflow DAGs — same demo-data volume as Jupyter
RAG_TABLE_PATH = os.environ.get("AILAKE_RAG_TABLE_PATH", "/data/ailake_rag_airflow")
DIM = int(os.environ.get("DEMO_DIM", "32"))

# Basic-auth header
AUTH_HEADER = b"Basic " + b64encode(f"{AIRFLOW_USER}:{AIRFLOW_PASS}".encode())

def af_get(path: str) -> dict:
    """GET /api/v1/<path>, return parsed JSON."""
    req = urlrequest.Request(
        f"{AIRFLOW_URL}/api/v1/{path}",
        headers={"Authorization": AUTH_HEADER, "Content-Type": "application/json"},
    )
    with urlrequest.urlopen(req, timeout=10) as resp:
        return json.loads(resp.read())

def af_post(path: str, body: dict) -> dict:
    """POST /api/v1/<path> with JSON body, return parsed JSON."""
    data = json.dumps(body).encode()
    req = urlrequest.Request(
        f"{AIRFLOW_URL}/api/v1/{path}",
        data=data,
        headers={"Authorization": AUTH_HEADER, "Content-Type": "application/json"},
        method="POST",
    )
    with urlrequest.urlopen(req, timeout=10) as resp:
        return json.loads(resp.read())

print(f"Airflow URL : {AIRFLOW_URL}")
print(f"RAG table   : {RAG_TABLE_PATH}")

## 1. Health check

Confirms Airflow is up and both the scheduler and metadata DB are healthy.
If this cell raises `URLError`, start the Airflow service:

```bash
docker compose -f tests/docker/compose-demo.yml --profile airflow up -d
```

In [ ]:
try:
    health = af_get("health")
    print(json.dumps(health, indent=2))
    meta_status    = health.get("metadatabase", {}).get("status", "unknown")
    sched_status   = health.get("scheduler",    {}).get("status", "unknown")
    print(f"\nMetadata DB : {meta_status}")
    print(f"Scheduler   : {sched_status}")
    AIRFLOW_UP = True
except Exception as exc:
    print(f"Airflow not reachable: {exc}")
    print("Cells 2-8 will be skipped. Section 9 works without Airflow.")
    AIRFLOW_UP = False

## 2. List DAGs

Two demo DAGs are pre-loaded from `tests/docker/demo/dags/` (hot-reloaded, read-only mount):

| DAG | Schedule | Purpose |
|---|---|---|
| `ailake_ingest_search` | `@daily` | Write 50 docs → vector search → FTS → assemble context |
| `ailake_compaction` | `@weekly` | Compact small files + print table metadata |

In [ ]:
if not AIRFLOW_UP:
    print("SKIP — Airflow not running")
else:
    dags = af_get("dags")["dags"]
    ailake_dags = [d for d in dags if d["dag_id"].startswith("ailake_")]
    print(f"Total DAGs: {len(dags)}  |  AI-Lake DAGs: {len(ailake_dags)}")
    for d in ailake_dags:
        state = "PAUSED" if d["is_paused"] else "active"
        print(f"  {d['dag_id']:35s}  schedule={d.get('schedule_interval',{}).get('value','?'):10s}  [{state}]")

## 3. Trigger `ailake_ingest_search`

Unpause the DAG (paused on first load) and trigger a manual run via the REST API.
The run ID is stored for polling in the next cell.

In [ ]:
RUN_ID = None

if not AIRFLOW_UP:
    print("SKIP — Airflow not running")
else:
    DAG_ID = "ailake_ingest_search"

    # Unpause so the scheduler picks it up
    try:
        import urllib.request as urllib_req
        data = json.dumps({"is_paused": False}).encode()
        req = urllib_req.Request(
            f"{AIRFLOW_URL}/api/v1/dags/{DAG_ID}",
            data=data,
            headers={"Authorization": AUTH_HEADER, "Content-Type": "application/json"},
            method="PATCH",
        )
        with urllib_req.urlopen(req, timeout=10):
            pass
        print(f"DAG '{DAG_ID}' unpaused.")
    except Exception as exc:
        print(f"unpause: {exc} (may already be active)")

    run_conf = {
        "dag_run_id": f"notebook_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')}",
        "conf": {},
    }
    run = af_post(f"dags/{DAG_ID}/dagRuns", run_conf)
    RUN_ID = run["dag_run_id"]
    print(f"Triggered run  : {RUN_ID}")
    print(f"State          : {run['state']}")
    print(f"Execution date : {run['execution_date']}")

## 4. Poll until run completes

Polls every 5 s for up to 3 minutes. With SequentialExecutor all tasks run
sequentially in the scheduler process — expected total time ~10-20 s.

In [ ]:
FINAL_STATE = None

if not AIRFLOW_UP or RUN_ID is None:
    print("SKIP")
else:
    for attempt in range(36):   # 36 × 5 s = 3 min
        run_info = af_get(f"dags/{DAG_ID}/dagRuns/{RUN_ID}")
        state    = run_info["state"]
        print(f"  [{attempt * 5:3d}s]  state={state}", flush=True)
        if state in ("success", "failed"):
            FINAL_STATE = state
            break
        time.sleep(5)
    else:
        print("Timed out — check the Airflow UI at http://localhost:8090")

    if FINAL_STATE:
        print(f"\nRun completed: {FINAL_STATE.upper()}")

## 5. Inspect task logs

Fetch the log for the `write_docs` task from the completed run.
Airflow stores task logs in the container; the REST API streams them as plain text.

In [ ]:
if not AIRFLOW_UP or RUN_ID is None:
    print("SKIP")
else:
    for task_id in ["write_docs", "vector_search", "fts_search"]:
        print(f"── Task: {task_id} ─────────────────")
        try:
            log_path = f"dags/{DAG_ID}/dagRuns/{RUN_ID}/taskInstances/{task_id}/logs/1"
            req = urlrequest.Request(
                f"{AIRFLOW_URL}/api/v1/{log_path}",
                headers={"Authorization": AUTH_HEADER, "Accept": "text/plain"},
            )
            with urlrequest.urlopen(req, timeout=10) as resp:
                log_text = resp.read().decode(errors="replace")
            # Print last 15 lines (skip Airflow boilerplate at the top)
            lines = [l for l in log_text.splitlines() if l.strip()]
            for line in lines[-15:]:
                print(" ", line)
        except Exception as exc:
            print(f"  (log unavailable: {exc})")
        print()

## 6. Pull XCom results

Tasks push their return values to XCom automatically (TaskFlow API).
Pull the `vector_search` and `fts_search` results from the completed run.

In [ ]:
import pandas as pd

if not AIRFLOW_UP or RUN_ID is None:
    print("SKIP")
else:
    for task_id in ["vector_search", "fts_search"]:
        print(f"── XCom from {task_id} ─────────────")
        try:
            xcom = af_get(
                f"dags/{DAG_ID}/dagRuns/{RUN_ID}/taskInstances/{task_id}/xcomEntries/return_value"
            )
            value = xcom.get("value")
            rows  = json.loads(value) if isinstance(value, str) else value
            if rows:
                df = pd.DataFrame(rows)
                print(df.head(5).to_string(index=False))
            else:
                print(f"  (empty result)")
        except Exception as exc:
            print(f"  (xcom unavailable: {exc})")
        print()

## 7. Read Airflow-written data in Jupyter

The Airflow container and the Jupyter container share the `demo-data` Docker
volume — data written by Airflow is immediately available here via `ailake.search()`.

In [ ]:
import pathlib

if not pathlib.Path(RAG_TABLE_PATH).exists():
    print(f"Table not found at {RAG_TABLE_PATH}.")
    print("Run the DAG first (section 3) or check --profile airflow is active.")
else:
    rng   = np.random.default_rng(99)
    query = rng.standard_normal(DIM).astype(np.float32)
    query /= np.linalg.norm(query)

    results = ailake.search(RAG_TABLE_PATH, query.tolist(), top_k=5, fetch_data=True).to_pandas()
    print(f"Vector search over Airflow-written table ({RAG_TABLE_PATH}):")
    print(results[["text", "_distance"]].to_string(index=False))
    print()

    fts_hits = ailake.search_text(RAG_TABLE_PATH, "retrieval augmented generation", top_k=3)
    print("FTS search (Tantivy) — 'retrieval augmented generation':")
    for h in fts_hits:
        print(f"  row_id={h.get('row_id')}  score={h.get('score', 0):.4f}")

## 8. Trigger `ailake_compaction`

`ailake_compaction` is a maintenance DAG — scheduled weekly but triggered here
manually to show the compaction + metadata log pattern.

In [ ]:
if not AIRFLOW_UP:
    print("SKIP — Airflow not running")
else:
    COMPACT_DAG = "ailake_compaction"

    # Unpause
    try:
        data = json.dumps({"is_paused": False}).encode()
        req  = urlrequest.Request(
            f"{AIRFLOW_URL}/api/v1/dags/{COMPACT_DAG}",
            data=data,
            headers={"Authorization": AUTH_HEADER, "Content-Type": "application/json"},
            method="PATCH",
        )
        with urlrequest.urlopen(req, timeout=10):
            pass
    except Exception:
        pass

    run_id_ts = f"notebook_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')}"
    run = af_post(f"dags/{COMPACT_DAG}/dagRuns", {"dag_run_id": run_id_ts, "conf": {}})
    compact_run_id = run["dag_run_id"]
    print(f"Triggered compaction run: {compact_run_id}")

    for attempt in range(24):
        info  = af_get(f"dags/{COMPACT_DAG}/dagRuns/{compact_run_id}")
        state = info["state"]
        print(f"  [{attempt * 5:3d}s]  {state}", flush=True)
        if state in ("success", "failed"):
            break
        time.sleep(5)

    print(f"\nCompaction run: {state.upper()}")

## 9. Direct PythonOperator demo — no Airflow required

Instantiate and run tasks directly by calling `.execute()` with a minimal mock context.
Useful for unit-testing DAG logic or running AI-Lake pipelines without an Airflow scheduler.

> This is how the demo DAGs are structured internally — each `@task` function is
> a plain Python callable. Airflow wraps them; you can also call them directly.

In [ ]:
import os
import numpy as np
import ailake

DIRECT_PATH = "/data/ailake_direct_demo"

# ── Step 1: write_docs ────────────────────────────────────────────────────────
def write_docs_fn(path: str, dim: int = 32) -> str:
    rng    = np.random.default_rng(42)
    topics = ["transformer", "vector search", "Iceberg", "RAG", "embedding"]
    texts  = [f"Direct demo doc {i}: about {topics[i % len(topics)]}." for i in range(20)]
    embs   = rng.standard_normal((20, dim)).astype(np.float32)
    embs  /= np.linalg.norm(embs, axis=1, keepdims=True)
    table  = ailake.open_table(path, dim=dim, metric="cosine", fts_text_columns=["text"])
    table.insert(texts, embs)
    snap_id = table.commit()
    return str(snap_id)

# ── Step 2: vector_search ─────────────────────────────────────────────────────
def vector_search_fn(path: str, dim: int = 32) -> list:
    rng   = np.random.default_rng(7)
    query = rng.standard_normal(dim).astype(np.float32)
    query /= np.linalg.norm(query)
    return ailake.search(path, query.tolist(), top_k=3).to_list()

# ── Step 3: fts_search ────────────────────────────────────────────────────────
def fts_search_fn(path: str) -> list:
    return ailake.search_text(path, "vector search Iceberg", top_k=3)

# Run the pipeline directly
snap   = write_docs_fn(DIRECT_PATH, dim=DIM)
vec    = vector_search_fn(DIRECT_PATH, dim=DIM)
fts    = fts_search_fn(DIRECT_PATH)

print(f"Snapshot       : {snap}")
print(f"Vector top-3   : {[(r['row_id'], round(r['distance'], 4)) for r in vec]}")
print(f"FTS top-3 IDs  : {[h.get('row_id') for h in fts]}")

## 10. `AilakeWriteOperator` — production CLI-based pattern

For production environments where the `ailake` CLI binary is available on PATH,
use the `airflow-providers-ailake` operators instead of plain PythonOperator:

```python
from airflow_providers_ailake.operators.ailake import (
    AilakeWriteOperator,
    AilakeCompactOperator,
    AilakeSearchOperator,
    AilakeFtsSearchOperator,
)

# Write + FTS index
write = AilakeWriteOperator(
    task_id="write",
    table="default.docs",
    source_file="{{ ti.xcom_pull('generate') }}",
    embeddings_column="embedding",
    fts_columns=["chunk_text", "document_title"],
    pre_normalize=True,
    deferred=True,                   # ~200k vec/s — index builds in background
    ailake_conn_id="ailake_default",
)

# Vector search — results pushed to XCom key 'search_results'
search = AilakeSearchOperator(
    task_id="search",
    table="default.docs",
    query_xcom_task_id="embed_query",  # embed step pushes vector to XCom
    top_k=20,
    hybrid_text="{{ dag_run.conf['query'] }}",
    bm25_weight=0.4,
)

# FTS-only search — results pushed to XCom key 'fts_results'
fts = AilakeFtsSearchOperator(
    task_id="fts",
    table="default.docs",
    query_text="{{ dag_run.conf['query'] }}",
    text_columns=["chunk_text", "document_title"],
    top_k=10,
)

# Compact weekly
compact = AilakeCompactOperator(
    task_id="compact",
    table="default.docs",
    min_files=4,
)

write >> [search, fts] >> compact
```

**Connection setup** (Airflow UI → Admin → Connections):

| Field | Value |
|---|---|
| Connection ID | `ailake_default` |
| Connection Type | `ailake` |
| Host | `s3://my-bucket/warehouse` (or local path) |
| Extra (JSON) | `{"aws_access_key_id": "...", "aws_secret_access_key": "...", "aws_region": "us-east-1"}` |

The hook injects credentials into the subprocess environment — S3, GCS, and Azure are all supported.

In [ ]:
# Verify the provider is importable in this Jupyter environment.
# (It is installed in the Airflow container; for Jupyter, install manually if needed.)
try:
    from airflow_providers_ailake.operators.ailake import (
        AilakeWriteOperator,
        AilakeSearchOperator,
        AilakeFtsSearchOperator,
        AilakeCompactOperator,
    )
    from airflow_providers_ailake.hooks.ailake import AilakeHook
    print("airflow-providers-ailake imported successfully.")
    print(f"  AilakeWriteOperator     : {AilakeWriteOperator}")
    print(f"  AilakeFtsSearchOperator : {AilakeFtsSearchOperator}")
    print(f"  AilakeHook              : {AilakeHook}")
except ImportError as exc:
    print(f"Provider not installed in Jupyter: {exc}")
    print("To install: pip install <repo>/airflow-providers-ailake")

## Next Steps

| Resource | Location |
|---|---|
| Airflow provider source | `airflow-providers-ailake/` |
| DAG examples | `tests/docker/demo/dags/` |
| Provider README | `airflow-providers-ailake/README.md` |
| Python API reference | `ailake-py/README.md` |
| Tantivy FTS deep-dive | `11_fts.ipynb` |
| Hybrid BM25+vector | `09_hybrid_search.ipynb` |
| Agent memory pipelines | `08_agents.ipynb` |